## Import libraries

In [44]:
import numpy as np
import pandas as pd
from glob import glob
from obspy import read, UTCDateTime as utc
from datetime import datetime, timezone
from obspy.geodetics.base import gps2dist_azimuth
from tqdm import tqdm

## Load `PyOcto` output files

### Event table

In [13]:
# event_table_251101=pd.read_csv("/home/msseo/works/72.2025_Imsil_swarm/4.PyOcto/251101/pyocto_kim1983_prob0.5.csv", sep=",")
event_table_251102=pd.read_csv("/home/msseo/works/72.2025_Imsil_swarm/4.PyOcto/251102/pyocto_kim1983_prob0.5.csv", sep=",")
# event_table_251103=pd.read_csv("/home/msseo/works/72.2025_Imsil_swarm/4.PyOcto/251103/pyocto_kim1983_prob0.5.csv", sep=",")
event_table_combined=event_table_251102
# event_table_combined=pd.concat([event_table_251101, event_table_251102, event_table_251103]).reset_index(drop=True)
event_table_combined.head()

,idx,time,x,y,z,picks,latitude,longitude,depth
0,0,2025-11-02 03:06:07.098809+00:00,5.883866,6.648589,22.421875,9,35.559907,127.314899,22.421875
1,1,2025-11-02 04:13:26.740459+00:00,12.302628,-30.710996,9.296875,14,35.223117,127.385135,9.296875
2,2,2025-11-02 06:20:48.101702+00:00,11.232835,-5.514996,4.921875,8,35.450229,127.373730,4.921875
3,3,2025-11-02 07:36:46.580163+00:00,1.604691,12.730382,30.078125,8,35.614738,127.267712,30.078125
4,4,2025-11-02 10:42:33.640302+00:00,-15.512010,3.173279,0.546875,8,35.528480,127.078970,0.546875


In [17]:
len(event_table_combined)

39

### Pick assignment table

In [14]:
# pick_table_251101=pd.read_csv("/home/msseo/works/72.2025_Imsil_swarm/4.PyOcto/251101/pyocto_assignment_kim1983_prob0.5.csv", sep=",")
pick_table_251102=pd.read_csv("/home/msseo/works/72.2025_Imsil_swarm/4.PyOcto/251102/pyocto_assignment_kim1983_prob0.5.csv", sep=",")
# pick_table_251103=pd.read_csv("/home/msseo/works/72.2025_Imsil_swarm/4.PyOcto/251103/pyocto_assignment_kim1983_prob0.5.csv", sep=",")
pick_table_combined=pick_table_251102
# pick_table_combined=pd.concat([pick_table_251101, pick_table_251102, pick_table_251103]).reset_index(drop=True)
pick_table_combined.head()

,event_idx,pick_idx,residual,station,phase,time
0,0,393,-1.205177,KS.CHOA.,S,1.762053e+09
1,0,790,0.951169,KS.GKSA.,P,1.762053e+09
2,0,1728,-0.514246,KS.GKSA.,S,1.762053e+09
3,0,2521,0.930973,KS.IMSB.,P,1.762053e+09
4,0,2874,0.680056,KS.IMSB.,S,1.762053e+09


## Write `HYPOINVERSE` `PHS` file

### Function to convert degrees to degrees/minutes/seconds

In [15]:
def degreetominute(angle):
    angle_deg=int(angle)
    angle_min=int(100*60*(angle-int(angle)))
    
    return [angle_deg, angle_min]

### Write input file

In [58]:
# Only change the names of following path and filename (do not need to touch rest of the code)
input_file="/home/msseo/works/72.2025_Imsil_swarm/5.Detection_HypoInv/PHS/Imsil.phs"

############### Keep codes below same regardless of the files that were used for picking ###############
f=open(input_file, 'w')
id_number=0
for i in tqdm(range(len(event_table_combined))):
    eventid=event_table_combined["idx"][i]
    origin_time=utc(event_table_combined["time"][i].split("+")[0])
    year=str(origin_time.year)
    month=str(origin_time.month).zfill(2)
    day=str(origin_time.day).zfill(2)
    hour=str(origin_time.hour).zfill(2)
    minute=str(origin_time.minute).zfill(2)
    second=str(origin_time.second).zfill(2)
    millisecond=str(int(origin_time.microsecond/1000)).zfill(2)
    
    catalog_eventid=str(year)+str(month).zfill(2)+str(day).zfill(2)+str(hour).zfill(2)+str(minute).zfill(2)+str(second).zfill(2)

    # Write event origin time / latitude / longitude
    lat_deg, lat_min=str(degreetominute(event_table_combined["latitude"][i])[0]), str(degreetominute(event_table_combined["latitude"][i])[1]).zfill(4)
    lon_deg, lon_min=str(degreetominute(event_table_combined["longitude"][i])[0]), str(degreetominute(event_table_combined["longitude"][i])[1]).zfill(4)
    eventinfo=catalog_eventid+millisecond+lat_deg+"N"+lat_min+lon_deg+"E"+lon_min

    f.write(eventinfo+"\n")

    # Get phase arrival information
    event_pick_table=pick_table_combined[pick_table_combined["event_idx"]==eventid].reset_index(drop=True)
    
    for j in range(len(event_pick_table)):
        pick_time=utc(datetime.fromtimestamp(event_pick_table["time"][j], tz=timezone.utc))
        network=event_pick_table["station"][j].split(".")[0]
        station_name=event_pick_table["station"][j].split(".")[1]
        phase=event_pick_table["phase"][j]
        
        if phase=="P":
            channel="HHZ"
            p_weight="0"
        
            f.write(station_name.ljust(5))
            f.write(network.ljust(4))
            f.write(channel[:3].ljust(4))
            f.write("IP".ljust(3))
            f.write(p_weight)
            f.write(str(pick_time.year))
            f.write(str(pick_time.month).zfill(2))
            f.write(str(pick_time.day).zfill(2))
            f.write(str(pick_time.hour).zfill(2))
            f.write(str(pick_time.minute).zfill(2).ljust(3))
            f.write(str(pick_time.second).zfill(2))
            f.write(str(pick_time.microsecond).zfill(6)[:2])
            f.write("\n")
            
        elif phase=="S":
            channel="HHN"
            s_weight="1"

            f.write(station_name.ljust(5))
            f.write(network.ljust(4))
            f.write(channel.ljust(4))
            f.write("    ") # P wave phase name & weight missing
            f.write(str(pick_time.year))
            f.write(str(pick_time.month).zfill(2))
            f.write(str(pick_time.day).zfill(2))
            f.write(str(pick_time.hour).zfill(2))
            f.write(str(pick_time.minute).zfill(2).ljust(15))
            f.write(str(pick_time.second).zfill(2))
            f.write(str(pick_time.microsecond).zfill(6)[:2])
            f.write("ES".ljust(3))
            f.write(s_weight)
            f.write("\n")

    f.write(" "*66+"200"+f"{id_number}".zfill(3))
    f.write("\n")
    id_number += 1
            
f.close()

100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████| 39/39 [00:00<00:00, 876.14it/s]
